# __Wolf Sheep Predation Model__

_Kelby Mace 03/27/2026_

In [ ]:
import random
import math
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


@dataclass
class Patch:
    x: int
    y: int
    color: str = "green"      # "green" or "brown"
    countdown: int = 0


class Animal:
    def __init__(self, model, x, y, energy):
        self.model = model
        self.x = x
        self.y = y
        self.energy = energy
        self.heading = random.uniform(0, 360)
        self.alive = True
        self.label = ""

    def move(self):
        self.heading += random.randint(0, 49)
        self.heading -= random.randint(0, 49)

        radians = math.radians(self.heading)
        dx = round(math.cos(radians))
        dy = round(math.sin(radians))

        self.x = (self.x + dx) % self.model.width
        self.y = (self.y + dy) % self.model.height

    def die(self):
        self.alive = False

    def death(self):
        if self.energy < 0:
            self.die()


class Sheep(Animal):
    def eat_grass(self):
        patch = self.model.get_patch(self.x, self.y)
        if patch.color == "green":
            patch.color = "brown"
            self.energy += self.model.sheep_gain_from_food

    def reproduce(self):
        if random.random() * 100 < self.model.sheep_reproduce:
            self.energy /= 2
            child = Sheep(self.model, self.x, self.y, self.energy)
            child.heading = self.heading + random.uniform(0, 360)
            child.move()
            self.model.new_sheep.append(child)

    def step(self):
        self.move()

        if self.model.model_version == "sheep-wolves-grass":
            self.energy -= 1
            self.eat_grass()
            self.death()

        if self.alive:
            self.reproduce()


class Wolf(Animal):
    def eat_sheep(self):
        sheep_here = [
            s for s in self.model.sheep
            if s.alive and s.x == self.x and s.y == self.y
        ]
        if sheep_here:
            prey = random.choice(sheep_here)
            prey.die()
            self.energy += self.model.wolf_gain_from_food

    def reproduce(self):
        if random.random() * 100 < self.model.wolf_reproduce:
            self.energy /= 2
            child = Wolf(self.model, self.x, self.y, self.energy)
            child.heading = self.heading + random.uniform(0, 360)
            child.move()
            self.model.new_wolves.append(child)

    def step(self):
        self.move()
        self.energy -= 1
        self.eat_sheep()
        self.death()

        if self.alive:
            self.reproduce()


class WolfSheepModel:
    def __init__(
        self,
        width=50,
        height=50,
        initial_number_sheep=100,
        initial_number_wolves=50,
        sheep_gain_from_food=4,
        wolf_gain_from_food=20,
        sheep_reproduce=4.0,
        wolf_reproduce=5.0,
        grass_regrowth_time=30,
        model_version="sheep-wolves-grass",
        show_energy=False,
        max_sheep=30000,
    ):
        self.width = width
        self.height = height

        self.initial_number_sheep = initial_number_sheep
        self.initial_number_wolves = initial_number_wolves
        self.sheep_gain_from_food = sheep_gain_from_food
        self.wolf_gain_from_food = wolf_gain_from_food
        self.sheep_reproduce = sheep_reproduce
        self.wolf_reproduce = wolf_reproduce
        self.grass_regrowth_time = grass_regrowth_time
        self.model_version = model_version
        self.show_energy = show_energy
        self.max_sheep = max_sheep

        self.patches = []
        self.sheep = []
        self.wolves = []
        self.new_sheep = []
        self.new_wolves = []
        self.ticks = 0

    def get_patch(self, x, y):
        return self.patches[y][x]

    def setup(self):
        self.ticks = 0
        self.sheep = []
        self.wolves = []
        self.new_sheep = []
        self.new_wolves = []

        self.patches = []
        for y in range(self.height):
            row = []
            for x in range(self.width):
                patch = Patch(x, y)

                if self.model_version == "sheep-wolves-grass":
                    patch.color = random.choice(["green", "brown"])
                    if patch.color == "green":
                        patch.countdown = self.grass_regrowth_time
                    else:
                        patch.countdown = random.randrange(self.grass_regrowth_time)
                else:
                    patch.color = "green"

                row.append(patch)
            self.patches.append(row)

        for _ in range(self.initial_number_sheep):
            x = random.randrange(self.width)
            y = random.randrange(self.height)
            energy = random.randrange(2 * self.sheep_gain_from_food)
            self.sheep.append(Sheep(self, x, y, energy))

        for _ in range(self.initial_number_wolves):
            x = random.randrange(self.width)
            y = random.randrange(self.height)
            energy = random.randrange(2 * self.wolf_gain_from_food)
            self.wolves.append(Wolf(self, x, y, energy))

        self.display_labels()

    def grow_grass(self):
        for row in self.patches:
            for patch in row:
                if patch.color == "brown":
                    if patch.countdown <= 0:
                        patch.color = "green"
                        patch.countdown = self.grass_regrowth_time
                    else:
                        patch.countdown -= 1

    def grass(self):
        if self.model_version == "sheep-wolves-grass":
            return [
                patch
                for row in self.patches
                for patch in row
                if patch.color == "green"
            ]
        return 0

    def display_labels(self):
        for s in self.sheep:
            s.label = ""
        for w in self.wolves:
            w.label = ""

        if self.show_energy:
            for w in self.wolves:
                w.label = str(round(w.energy))
            if self.model_version == "sheep-wolves-grass":
                for s in self.sheep:
                    s.label = str(round(s.energy))

    def go(self):
        if not self.sheep and not self.wolves:
            return False

        if not self.wolves and len(self.sheep) > self.max_sheep:
            print("The sheep have inherited the earth")
            return False

        self.new_sheep = []
        self.new_wolves = []

        for sheep in self.sheep[:]:
            if sheep.alive:
                sheep.step()

        self.sheep = [s for s in self.sheep if s.alive]

        for wolf in self.wolves[:]:
            if wolf.alive:
                wolf.step()

        self.wolves = [w for w in self.wolves if w.alive]
        self.sheep = [s for s in self.sheep if s.alive]

        self.sheep.extend(self.new_sheep)
        self.wolves.extend(self.new_wolves)

        if self.model_version == "sheep-wolves-grass":
            self.grow_grass()

        self.ticks += 1
        self.display_labels()
        return True

    def count_sheep(self):
        return len(self.sheep)

    def count_wolves(self):
        return len(self.wolves)

    def patch_array(self):
        """
        Convert patches into a numeric grid for plotting:
        green grass = 1
        brown/no grass = 0
        """
        arr = np.zeros((self.height, self.width))
        for y in range(self.height):
            for x in range(self.width):
                arr[y, x] = 1 if self.patches[y][x].color == "green" else 0
        return arr

In [4]:
def animate_model(model, steps=200, interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))

    grass_img = ax.imshow(
        model.patch_array(),
        origin="lower",
        interpolation="nearest",
        cmap="YlGn",
        vmin=0,
        vmax=1
    )

    sheep_scatter = ax.scatter([], [], s=25, marker="o", label="Sheep")
    wolf_scatter = ax.scatter([], [], s=40, marker="s", label="Wolves")

    title = ax.set_title("")
    ax.set_xlim(-0.5, model.width - 0.5)
    ax.set_ylim(-0.5, model.height - 0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(loc="upper right")

    anim = None

    def update(frame):
        running = model.go()

        grass_img.set_data(model.patch_array())

        sheep_xy = np.array([(s.x, s.y) for s in model.sheep]) if model.sheep else np.empty((0, 2))
        wolf_xy = np.array([(w.x, w.y) for w in model.wolves]) if model.wolves else np.empty((0, 2))

        sheep_scatter.set_offsets(sheep_xy)
        wolf_scatter.set_offsets(wolf_xy)

        title.set_text(
            f"Tick {model.ticks} | Sheep: {model.count_sheep()} | Wolves: {model.count_wolves()}"
        )

        if not running and anim is not None:
            anim.event_source.stop()

        return grass_img, sheep_scatter, wolf_scatter, title

    anim = FuncAnimation(
        fig,
        update,
        frames=steps,
        interval=interval,
        blit=False,
        repeat=False
    )

    return anim

In [ ]:
from IPython.display import HTML

model = WolfSheepModel(
    width=25,
    height=25,
    initial_number_sheep=10,
    initial_number_wolves=2,
    model_version="sheep-wolves-grass",
)
model.setup()

anim = animate_model(model, steps=100, interval=120)
plt.close(anim._fig)
HTML(anim.to_jshtml())